In [1]:
!pip install sentence-transformers

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached pillow-12.0.0-cp312-cp312-win_amd64.whl.metadata (9.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached numpy-2.3.5-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
  Using cached charset_normalizer-3.4.4-cp312-cp312-win_amd64.whl.metadata (38 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached certifi-2025.11.12-py3-none-any.whl.metadata (2.5 kB)
  Using cached anyio-4.12.0-py3-non


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!pip install pinecone

  Using cached orjson-3.11.4-cp312-cp312-win_amd64.whl.metadata (42 kB)
   ---------------------------------------- 0.0/745.9 kB ? eta -:--:--
   --------------------------------------- 745.9/745.9 kB 10.3 MB/s eta 0:00:00
Using cached orjson-3.11.4-cp312-cp312-win_amd64.whl (131 kB)
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install python-dotenv

  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

c:\Users\kshit\OneDrive\Desktop\CodingNinjasAICourse\Rag\Vector Databases\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
pinecone = Pinecone(api_key=os.getenv("PINECONE_KEY"))

In [7]:
pinecone

In [8]:
INDEX_NAME = "ragtest"

In [9]:
pinecone.list_indexes()

[]

In [10]:
existing_indexes = [idx["name"] for idx in pinecone.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pinecone.create_index(
        name=INDEX_NAME,
        dimension=384,          # 384-dim for MiniLM
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )
    print(f"✅ Created index: {INDEX_NAME}")
else:
    print(f"ℹ️ Index already exists: {INDEX_NAME}")

✅ Created index: ragtest


In [11]:
model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\kshit\OneDrive\Desktop\CodingNinjasAICourse\Rag\Vector Databases\env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kshit\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xe

In [16]:
docs = [
    {"id": "doc1", "text": "Pandas is a Python library for data analysis."},
    {"id": "doc2", "text": "Pinecone is a vector database for semantic search."},
    {"id": "doc3", "text": "Spark enables distributed data processing."},
]


embeddings = model.encode([d["text"] for d in docs])

print(embeddings)

[[-0.0234553  -0.0484851  -0.08455052 ...  0.11974922  0.07560261
  -0.00940559]
 [ 0.04939834 -0.10375695 -0.04593762 ... -0.01025995  0.01978335
   0.00459547]
 [-0.04072915  0.00043279 -0.00490035 ... -0.01421899 -0.00235178
  -0.00569165]]


In [17]:
embeddings[0].shape

(384,)

In [18]:
vectors = [
    {
        "id": d["id"],
        "values" : emb.tolist(),
        "metadata": {
            "text": d["text"]
        },
    }
    for d, emb in zip(docs, embeddings)
]

In [19]:
vectors

[{'id': 'doc1',
  'values': [-0.023455295711755753,
   -0.04848510026931763,
   -0.08455052226781845,
   -0.012492821551859379,
   0.02092602290213108,
   -0.1094793975353241,
   -0.023819997906684875,
   -0.01498094666749239,
   -0.041525982320308685,
   0.009696240536868572,
   0.05692518129944801,
   0.030102720484137535,
   -0.06993328034877777,
   -0.0005328971310518682,
   0.04045857489109039,
   0.019510863348841667,
   -0.06473543494939804,
   0.011390062980353832,
   0.07175992429256439,
   -0.13283824920654297,
   -0.03850864991545677,
   0.06506310403347015,
   -0.03946840763092041,
   0.022580234333872795,
   0.007656930014491081,
   0.007952059619128704,
   0.006368262693285942,
   -0.027092507109045982,
   -0.052588846534490585,
   0.02444557659327984,
   -0.054690148681402206,
   0.016791002824902534,
   0.03262421861290932,
   0.012354600243270397,
   -0.05021507665514946,
   0.009628030471503735,
   0.022557009011507034,
   0.034855470061302185,
   -0.04692231491208076